# Data Preparation

## 1. Load the Data
Load `weekly_route_operations.csv`

In [1]:
import pandas as pd

weekly_route_operations_df = pd.read_csv("../data/weekly_route_operations.csv")
weekly_route_operations_df.head()

,date,route_id,trade_volume_tonnes,shipping_delay_days,freight_cost_usd,container_availability_index,port_congestion_index,fuel_cost_index,commodity_price_index,weather_disruption_score,geopolitical_risk_score,route_status,carbon_emissions_tonnes
0,1/4/2015,R00001,8418.69,7.55,4586.66,71.42,82.22,62.48,19.20,59.51,57.15,Delayed,3681.49
1,1/11/2015,R00001,9343.00,9.33,4574.70,79.27,72.01,59.31,35.05,91.27,41.47,Delayed,4085.69
2,1/18/2015,R00001,7090.69,3.67,4520.46,60.16,44.64,63.24,74.22,46.69,79.69,Normal,3100.76
3,1/25/2015,R00001,7829.79,9.22,4696.95,66.43,97.82,67.62,68.59,56.39,67.15,Delayed,3423.97
4,2/1/2015,R00001,11339.59,6.36,4507.98,96.20,73.21,58.83,28.98,95.92,7.59,Delayed,4958.80


## 2. Fix Dates and perform sorting
`date` loads as text (e.g. `1/4/2015`), which doesn't sort in time order. Convert it to a real date and sort by route, then date.

In [2]:
weekly_route_operations_df["date"] = pd.to_datetime(weekly_route_operations_df["date"], format="%m/%d/%Y")
weekly_route_operations_df = weekly_route_operations_df.sort_values(["route_id", "date"]).reset_index(drop=True)

print("Date range:", weekly_route_operations_df["date"].min().date(), "to", weekly_route_operations_df["date"].max().date())

Date range: 2015-01-04 to 2026-12-27


## 3. Check Shape and Column Types
We expect there to be 31,300 rows and 13 columns

In [3]:
print("Shape:", weekly_route_operations_df.shape)
weekly_route_operations_df.info()

Shape: (31300, 13)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 31300 entries, 0 to 31299
Data columns (total 13 columns):
 #   Column                        Non-Null Count  Dtype         
---  ------                        --------------  -----         
 0   date                          31300 non-null  datetime64[ns]
 1   route_id                      31300 non-null  object        
 2   trade_volume_tonnes           31300 non-null  float64       
 3   shipping_delay_days           31300 non-null  float64       
 4   freight_cost_usd              31300 non-null  float64       
 5   container_availability_index  31300 non-null  float64       
 6   port_congestion_index         31300 non-null  float64       
 7   fuel_cost_index               31300 non-null  float64       
 8   commodity_price_index         31300 non-null  float64       
 9   weather_disruption_score      31300 non-null  float64       
 10  geopolitical_risk_score       31300 non-null  float64       
 11  route_sta

## 4. Missing Values and Duplicates
Each `(route_id, date)` pair should appear exactly once: 50 routes * 626 weeks.

In [4]:
print("Missing values:", weekly_route_operations_df.isna().sum().sum())
print("Duplicate rows:", weekly_route_operations_df.duplicated().sum())
print("Duplicate route-weeks:", weekly_route_operations_df.duplicated(["route_id", "date"]).sum())
print("Routes:", weekly_route_operations_df["route_id"].nunique(), "| Weeks:", weekly_route_operations_df["date"].nunique())

Missing values: 0
Duplicate rows: 0
Duplicate route-weeks: 0
Routes: 50 | Weeks: 626


## 5. Target Distribution
How often is each `route_status` seen?

In [8]:
route_status_counts = weekly_route_operations_df["route_status"].value_counts()
route_status_percent = (route_status_counts / len(weekly_route_operations_df) * 100).round(1)

pd.DataFrame({"count": route_status_counts, "percent": route_status_percent}).sort_values("count", ascending=False)

,count,percent
route_status,,
Delayed,20038,64.0
Normal,10264,32.8
Disrupted,998,3.2
